In [1]:
from pathlib import Path
import pandas as pd

# Esse procedimento abaixo é só pra poder alcançar os nossos dados, já que eles estão em uma pasta em um nível acima da pasta atual
BASE_DIR = Path.cwd().parent

DADOS_DIR = BASE_DIR / "dados" / "brutos"

arquivo_201 = DADOS_DIR / "RIPSA mrt2.01.csv"
arquivo_202 = DADOS_DIR / "RIPSA mrt2.02.csv"

# O 201 se trata dos dados relacionados à razão de mortalidade materna
# O 202 se trata dos dados de óbitos focados em causas e característicias demográficas
df_201 = pd.read_csv(arquivo_201)
df_202 = pd.read_csv(arquivo_202)

In [2]:
# Agora vou fazer a inspeção e as tratativas. Eu vou começar pela 201, obviamente

print(df_201.head())
print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
print(df_201.shape)
print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
print(df_201.dtypes)
print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
print(df_201.columns.to_list())

  Indicador   Ano  UF  Numerador - Obitos maternos  \
0  MRT.2.01  2009  11                         17.0   
1  MRT.2.01  2009  12                          7.0   
2  MRT.2.01  2009  13                         67.0   
3  MRT.2.01  2009  14                         10.0   
4  MRT.2.01  2009  15                         82.0   

   Numerador - Fator de correçao utilizado  \
0                                   1.5009   
1                                   1.0000   
2                                   1.2427   
3                                   1.2087   
4                                   1.2774   

   Numerador - Obitos maternos corrigido  Denominador - Nascidos vivos   Fator  
0                                25.5159                         26083  100000  
1                                 7.0000                         16892  100000  
2                                83.2622                         75729  100000  
3                                12.0865                          9673  10

In [3]:
# O ano já se encontra no tipo integer, o que cumpre com o que buscávamos

# Os dados do 202 só vão até 2024, enquanto os de 201 só vão até 2023. Por conta disto, optei por escolher até o ano de 2023 para a nossa análise, visto que ambos possuem informações
# Pelo número muito pequeno de dados existentes, preferimos optar por trazer todo o histórico. Apesar disto, removeremos o ano de 2024 e começaremos por 2009, para que ambos se encaixem dentro da faixa 2009-2023
print(sorted(df_201["Ano"].dropna().unique()))
print("\n-------------------------------------------\n")
print(sorted(df_202["Ano"].dropna().unique()))

df_202 = df_202[(df_202["Ano"] <= 2023) & (df_202["Ano"] >= 2009)].reset_index(drop=True)

[np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

-------------------------------------------

[np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [4]:
# Nota-se o valor fixo e constante de 100_000 na coluna fator. Entre criar uma variável única com o valor do fator e mantê-lo, eu optei por mantê-lo, pois documenta a escala do indicador

df_201["Fator"].value_counts(dropna=False)

Fator
100000    405
Name: count, dtype: int64

In [5]:
# O UF está como números. Acreditei ser a melhor opção, pela fonte de códigos da UF disponibilizada pelo IBGE, criar uma coluna com o estado em si.

estados_ibge = {
    11: "Rondônia",
    12: "Acre",
    13: "Amazonas",
    14: "Roraima",
    15: "Pará",
    16: "Amapá",
    17: "Tocantins",
    21: "Maranhão",
    22: "Piauí",
    23: "Ceará",
    24: "Rio Grande do Norte",
    25: "Paraíba",
    26: "Pernambuco",
    27: "Alagoas",
    28: "Sergipe",
    29: "Bahia",
    31: "Minas Gerais",
    32: "Espírito Santo",
    33: "Rio de Janeiro",
    35: "São Paulo",
    41: "Paraná",
    42: "Santa Catarina",
    43: "Rio Grande do Sul",
    50: "Mato Grosso do Sul",
    51: "Mato Grosso",
    52: "Goiás",
    53: "Distrito Federal"
}

df_201.insert(
    loc=3,
    column="Estado",
    value=df_201["UF"].map(estados_ibge)
)

In [6]:
df_201.head()

,Indicador,Ano,UF,Estado,Numerador - Obitos maternos,Numerador - Fator de correçao utilizado,Numerador - Obitos maternos corrigido,Denominador - Nascidos vivos,Fator
0,MRT.2.01,2009,11,Rondônia,17.0,1.5009,25.5159,26083,100000
1,MRT.2.01,2009,12,Acre,7.0,1.0000,7.0000,16892,100000
2,MRT.2.01,2009,13,Amazonas,67.0,1.2427,83.2622,75729,100000
3,MRT.2.01,2009,14,Roraima,10.0,1.2087,12.0865,9673,100000
4,MRT.2.01,2009,15,Pará,82.0,1.2774,104.7493,143140,100000


In [7]:
# Agora, vamos para o 202

print(df_202.head())
print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
print(df_202.shape)
print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
print(df_202.dtypes)
print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
print(df_202.columns.to_list())

  Indicador   Ano  Municipio  Faixa etária Cor ou raça  \
0  MRT.2.02  2009     110002  20 a 34 anos      Branca   
1  MRT.2.02  2009     110004  20 a 34 anos       Parda   
2  MRT.2.02  2009     110004  20 a 34 anos       Preta   
3  MRT.2.02  2009     110012  10 a 19 anos      Branca   
4  MRT.2.02  2009     110013  20 a 34 anos       Parda   

   Numerador - Causas obstétricas diretas  Numerador - Hipertensao  \
0                                       1                        0   
1                                       1                        0   
2                                       0                        0   
3                                       1                        1   
4                                       1                        0   

   Numerador - Hemorragia  Numerador - Infecçao puerperal  \
0                       0                               1   
1                       1                               0   
2                       0                      

In [8]:
# Conferindo os valores de município e anos disponíveis

print("Anos disponíveis:")
print(sorted(df_202["Ano"].unique()))

print("\nQuantidade de municípios:")
print(df_202["Municipio"].nunique())

print("\nValores de cor ou raça:")
print(df_202["Cor ou raça"].value_counts(dropna=False))

print("\nValores de faixa etária:")
print(df_202["Faixa etária"].value_counts(dropna=False))

Anos disponíveis:
[np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Quantidade de municípios:
3850

Valores de cor ou raça:
Cor ou raça
Parda             9708
Branca            6238
Preta             2311
~Nao informado     825
Indígena           397
Amarela             60
Name: count, dtype: int64

Valores de faixa etária:
Faixa etária
20 a 34 anos      11303
35 anos e mais     5419
10 a 19 anos       2813
~Nao informado        4
Name: count, dtype: int64


In [9]:
# Substituição básica apenas por questão de padrões

df_202["Cor ou raça"] = df_202["Cor ou raça"].replace(
    "~Nao informado", "Não informado"
)

df_202["Faixa etária"] = df_202["Faixa etária"].replace(
    "~Nao informado", "Não informado"
)

# Também coloquei município como texto, para que seja evitado qualquer conflito matemático na coluna

df_202["Municipio"] = (
    df_202["Municipio"]
    .astype("Int64")
    .astype("string")
)


In [12]:
# Salvando os códigos tratados.

DADOS_DIR = BASE_DIR / "dados" / "tratados"

df_201.to_csv(DADOS_DIR / "RIPSA 2.01 tratado.csv", index=False)
df_202.to_csv(DADOS_DIR / "RIPSA 2.02 tratado.csv", index=False)